In [8]:
import pandas as pd
import numpy as np
import os
print("1. O Python está olhando a partir da pasta:", os.getcwd())
print("2. O que tem dentro dessa pasta:", os.listdir('.'))

# 1. MAPEAMENTO DE CAMINHOS
# O notebook está em /notebooks, então voltamos um nível (../) para acessar /data
caminho_entrada = '../data/raw/T12 CONTROLE.xlsx'
caminho_saida = '../data/csv/T12_CONTROLE_convertido.csv'
pasta_saida = '../data/csv'

# Garantir que a pasta de destino exista antes de tentar salvar
if not os.path.exists(pasta_saida):
    os.makedirs(pasta_saida)
    print(f"Diretório criado: {pasta_saida}")

# 2. CARREGAMENTO DOS DADOS E DIAGNÓSTICO
print(f"Iniciando a leitura do arquivo: {caminho_entrada}")
print("Aguarde, arquivos .xlsx podem ser lentos para carregar...")

try:
    # Tentativa 1: Forçar o motor padrão de Excel (openpyxl)
    df = pd.read_excel(caminho_entrada, engine='openpyxl')
    print(f"\n✅ Leitura concluída como Excel! O arquivo contém {len(df)} linhas.")

except ValueError as e:
    print(f"\n❌ Erro de Valor retornado pelo pandas: {e}")
    print("\n🔍 Investigando a estrutura real do arquivo...")
    
    # Tentativa 2: Ler os primeiros 150 caracteres do arquivo como se fosse texto bruto
    try:
        with open(caminho_entrada, 'r', encoding='utf-8') as f:
            cabecalho_real = f.read(150)
            print("--- CONTEÚDO BRUTO DO ARQUIVO ---")
            print(cabecalho_real)
            print("---------------------------------")
            print("Diagnóstico: O seu arquivo provavelmente é um arquivo de texto (CSV/TSV) disfarçado de Excel.")
    except UnicodeDecodeError:
        print("Diagnóstico: O arquivo é binário, não é texto. Pode estar corrompido ou requer uma versão antiga do Excel (xls).")

# 3. APLICAÇÃO DO PROTOCOLO DE REQUISITOS (Verificação da Amostragem)
# A coluna 'Timegap (ms)' nos diz o tempo entre as capturas.
# Pelo nosso protocolo, gaps maiores que 100ms são considerados perdas de sinal/piscadas, 
# e não refletem a velocidade do hardware. Vamos isolar o comportamento normal.

# Filtra apenas os gaps menores ou iguais a 100ms
gaps_validos = df[df['Timegap (ms)'] <= 100]['Timegap (ms)']

media_gap_real = gaps_validos.mean()
# Calcula os Hz (1000 milissegundos divididos pelo tempo médio do gap)
taxa_hz_real = 1000 / media_gap_real 

print("\n--- 📊 RELATÓRIO DE QUALIDADE DO EQUIPAMENTO ---")
print("Equipamento: Eye-Tech TM3 (Nominal: 60 Hz)")
print(f"Média do gap de tempo (capturas normais): {media_gap_real:.2f} ms")
print(f"Taxa de amostragem REAL calculada: {taxa_hz_real:.2f} Hz")

# 4. EXPLORAÇÃO INICIAL DE RUÍDO
# Vamos olhar rapidamente para a coluna 'Detection' para entender como o tracker registra perdas.
print("\n--- ⚠️ ANÁLISE DE PERDA DE SINAL ---")
if 'Detection' in df.columns:
    print("Valores encontrados na coluna 'Detection' e suas quantidades:")
    print(df['Detection'].value_counts())
else:
    print("Atenção: Coluna 'Detection' não encontrada neste arquivo.")

# 5. CONVERSÃO E EXPORTAÇÃO
print("\nSalvando arquivo convertido...")
# Salvamos em CSV para acelerar todas as próximas etapas (index=False não cria coluna extra)
df.to_csv(caminho_saida, index=False)
print(f"✅ Conversão concluída com sucesso! Arquivo salvo em: {caminho_saida}")

1. O Python está olhando a partir da pasta: /workspaces/EyeTracking_analysis/notebooks
2. O que tem dentro dessa pasta: ['fase1_conversao.ipynb']
Iniciando a leitura do arquivo: ../data/raw/T12 CONTROLE.xlsx
Aguarde, arquivos .xlsx podem ser lentos para carregar...


BadZipFile: File is not a zip file

In [9]:
caminho_entrada = '../data/raw/T12 CONTROLE.xlsx'

print("🔍 Investigando a verdadeira estrutura do arquivo...\n")

try:
    # Vamos tentar forçar a leitura do arquivo como se ele fosse um arquivo de texto comum
    with open(caminho_entrada, 'r', encoding='utf-8') as f:
        cabecalho = f.read(500) # Lê os primeiros 500 caracteres
        print("✅ DIAGNÓSTICO: O arquivo é texto puro (CSV/TSV disfarçado).")
        print("\n--- INÍCIO DO CONTEÚDO BRUTO ---")
        print(cabecalho)
        print("--------------------------------")
except UnicodeDecodeError:
    print("⚠️ DIAGNÓSTICO: O arquivo é binário. Pode ser um formato muito antigo do Excel (.xls) ou foi corrompido no upload.")
    with open(caminho_entrada, 'rb') as f:
        print(f"Assinatura em bytes: {f.read(30)}")
except Exception as e:
    print(f"Erro inesperado: {e}")

🔍 Investigando a verdadeira estrutura do arquivo...

⚠️ DIAGNÓSTICO: O arquivo é binário. Pode ser um formato muito antigo do Excel (.xls) ou foi corrompido no upload.
Assinatura em bytes: b'\x00\x00\x00\x1cftypisom\x00\x00\x02\x00isomiso2mp41\x00\x00'


In [10]:
# 1. COLOQUE O NOME DO NOVO ARQUIVO AQUI
caminho_entrada = '../data/raw/T43 CONTROLE.xlsx'

print(f"🔍 Investigando a estrutura do arquivo: {caminho_entrada}\n")

try:
    # Tenta ler como texto (CSV/TSV disfarçado)
    with open(caminho_entrada, 'r', encoding='utf-8') as f:
        cabecalho = f.read(500)
        print("✅ SUCESSO: Este arquivo contém dados em formato de texto. Achamos o log!")
        print("\n--- INÍCIO DO CONTEÚDO BRUTO ---")
        print(cabecalho)
        print("--------------------------------")
except UnicodeDecodeError:
    # Se falhar, tenta ler os bytes para ver se é outro vídeo
    print("⚠️ DIAGNÓSTICO: Este arquivo também é binário (provavelmente outro vídeo).")
    with open(caminho_entrada, 'rb') as f:
        print(f"Assinatura em bytes: {f.read(30)}")
except FileNotFoundError:
    print("❌ ERRO: Arquivo não encontrado. Verifique se o nome do arquivo na linha 2 está escrito exatamente igual ao que está na pasta data/raw.")
except Exception as e:
    print(f"Erro inesperado: {e}")

🔍 Investigando a estrutura do arquivo: ../data/raw/T43 CONTROLE.xlsx

⚠️ DIAGNÓSTICO: Este arquivo também é binário (provavelmente outro vídeo).
Assinatura em bytes: b'PK\x03\x04\x14\x00\x08\x08\x08\x00IZ\x97\\\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x18\x00\x00\x00'


In [11]:
import pandas as pd
import numpy as np
import os

# 1. MAPEAMENTO DE CAMINHOS
caminho_entrada = '../data/raw/T43 CONTROLE.xlsx'
caminho_saida = '../data/csv/T43_CONTROLE_convertido.csv'
pasta_saida = '../data/csv'

# Garantir que a pasta de destino exista
if not os.path.exists(pasta_saida):
    os.makedirs(pasta_saida)

# 2. CARREGAMENTO DOS DADOS (Agora com o motor correto para arquivos PK/ZIP)
print(f"Iniciando a leitura do arquivo Excel verdadeiro: {caminho_entrada}")
print("Aguarde, extraindo dados do formato .xlsx...")

# Usamos o motor openpyxl, que é o padrão do Python para ler as estruturas ZIP do Excel
df = pd.read_excel(caminho_entrada, engine='openpyxl')
total_linhas = len(df)
print(f"\n✅ Leitura concluída! O arquivo contém {total_linhas} linhas de registro.")

# 3. APLICAÇÃO DO PROTOCOLO DE REQUISITOS (Verificação da Amostragem)
print("\n--- 📊 RELATÓRIO DE QUALIDADE DO EQUIPAMENTO ---")
if 'Timegap (ms)' in df.columns:
    # Filtramos gaps maiores que 100ms (piscadas/perdas) para achar o ritmo normal da máquina
    gaps_validos = df[df['Timegap (ms)'] <= 100]['Timegap (ms)']
    media_gap_real = gaps_validos.mean()
    taxa_hz_real = 1000 / media_gap_real 
    
    print("Equipamento: Eye-Tech TM3 (Nominal: 60 Hz)")
    print(f"Média do gap de tempo (capturas normais): {media_gap_real:.2f} ms")
    print(f"Taxa de amostragem REAL calculada: {taxa_hz_real:.2f} Hz")
else:
    print("Atenção: Coluna 'Timegap (ms)' não encontrada.")

# 4. EXPLORAÇÃO INICIAL DE RUÍDO
print("\n--- ⚠️ ANÁLISE DE PERDA DE SINAL ---")
# Na sua imagem, vi que existem hifens '-' na coluna AOI e valores 0 em coordenadas.
# Vamos mapear se há uma coluna específica de detecção ou contar as falhas de captura.
if 'Duration' in df.columns:
    zeros_duration = (df['Duration'] == 0).sum()
    print(f"Linhas com Duração = 0 (possível ausência de fixação/dado bruto cru): {zeros_duration}")

# 5. CONVERSÃO E EXPORTAÇÃO
print("\nSalvando arquivo convertido...")
df.to_csv(caminho_saida, index=False)
print(f"✅ Conversão concluída com sucesso! Arquivo otimizado salvo em: {caminho_saida}")

Iniciando a leitura do arquivo Excel verdadeiro: ../data/raw/T43 CONTROLE.xlsx
Aguarde, extraindo dados do formato .xlsx...

✅ Leitura concluída! O arquivo contém 11061 linhas de registro.

--- 📊 RELATÓRIO DE QUALIDADE DO EQUIPAMENTO ---
Equipamento: Eye-Tech TM3 (Nominal: 60 Hz)
Média do gap de tempo (capturas normais): 17.21 ms
Taxa de amostragem REAL calculada: 58.09 Hz

--- ⚠️ ANÁLISE DE PERDA DE SINAL ---
Linhas com Duração = 0 (possível ausência de fixação/dado bruto cru): 2

Salvando arquivo convertido...
✅ Conversão concluída com sucesso! Arquivo otimizado salvo em: ../data/csv/T43_CONTROLE_convertido.csv


In [12]:
import pandas as pd
import numpy as np

# 1. CARREGAMENTO DO ARQUIVO CONVERTIDO DA FASE 1
caminho_csv = '../data/csv/T43_CONTROLE_convertido.csv'
df = pd.read_csv(caminho_csv)

print("--- 🔍 MAPEAMENTO DE RUÍDOS E PISCADAS ---")

# 2. ANÁLISE DE COORDENADAS (Gaze X e Gaze Y)
# Contando valores zero, vazios (NaN) ou strings estranhas
zeros_x = (df['Gaze X'] == 0).sum()
zeros_y = (df['Gaze Y'] == 0).sum()
nulos_x = df['Gaze X'].isna().sum()

print(f"Coordenada Gaze X == 0: {zeros_x} linhas")
print(f"Coordenada Gaze Y == 0: {zeros_y} linhas")
print(f"Coordenadas Vazias (NaN): {nulos_x} linhas")

# 3. ANÁLISE DE PUPILA (Se houver diâmetro 0, o rastreador perdeu o olho)
if 'Pupil Diameter Left (mm)' in df.columns:
    zeros_pupila = (df['Pupil Diameter Left (mm)'] == 0).sum()
    nulos_pupila = df['Pupil Diameter Left (mm)'].isna().sum()
    print(f"\nDiâmetro Pupilar Esquerdo == 0: {zeros_pupila} linhas")
    print(f"Diâmetro Pupilar Vazio (NaN): {nulos_pupila} linhas")
else:
    print("\nAviso: Coluna 'Pupil Diameter Left (mm)' não encontrada com este nome exato.")

# 4. BUSCA POR CARACTERES ESPECIAIS (Hifens)
# Como vi hifens na sua captura de tela, vamos checar se eles poluem os dados
tem_hifen = df.apply(lambda col: col.astype(str).str.contains('-').any()).sum()
print(f"\nExistem {tem_hifen} colunas que contêm o caractere '-' (hífen).")

--- 🔍 MAPEAMENTO DE RUÍDOS E PISCADAS ---
Coordenada Gaze X == 0: 0 linhas
Coordenada Gaze Y == 0: 0 linhas
Coordenadas Vazias (NaN): 0 linhas

Diâmetro Pupilar Esquerdo == 0: 103 linhas
Diâmetro Pupilar Vazio (NaN): 0 linhas

Existem 4 colunas que contêm o caractere '-' (hífen).


In [13]:
import pandas as pd
import numpy as np

# 1. CARREGAMENTO DOS DADOS
caminho_csv = '../data/csv/T43_CONTROLE_convertido.csv'
caminho_saida = '../data/csv/T43_CONTROLE_limpo.csv'

print("Iniciando a Fase 2: Limpeza e Interpolação Fisiológica...")
df = pd.read_csv(caminho_csv)

# 2. SANITIZAÇÃO DE CARACTERES (Regra 1)
# O Pandas pode ter lido colunas com hífen '-' como texto. Vamos forçar a conversão 
# para NaN (Not a Number) para podermos fazer matemática com elas.
df.replace('-', np.nan, inplace=True)

# 3. INVALIDAÇÃO DE DADOS FALSOS (Regra 2)
col_pupila = 'Pupil Diameter Left (mm)'
# Garante que a coluna de pupila é lida como número
df[col_pupila] = pd.to_numeric(df[col_pupila], errors='coerce')

# Conta quantas linhas têm o olho perdido antes da limpeza
perdas_iniciais = (df[col_pupila] == 0).sum() + df[col_pupila].isna().sum()

# Onde a pupila for 0 ou NaN, transformamos X e Y em NaN (destruindo o dado falso)
mascara_perda = (df[col_pupila] == 0) | (df[col_pupila].isna())
df.loc[mascara_perda, ['Gaze X', 'Gaze Y']] = np.nan

# 4. INTERPOLAÇÃO LINEAR (Regras 3 e 4)
# Limite de 24 frames corresponde a ~413 ms na sua taxa real de 58 Hz.
# Qualquer buraco menor que 24 linhas será preenchido com uma reta ligando os pontos.
df['Gaze X'] = pd.to_numeric(df['Gaze X'], errors='coerce')
df['Gaze Y'] = pd.to_numeric(df['Gaze Y'], errors='coerce')

df['Gaze X'] = df['Gaze X'].interpolate(method='linear', limit=24, limit_direction='forward')
df['Gaze Y'] = df['Gaze Y'].interpolate(method='linear', limit=24, limit_direction='forward')

# 5. AUDITORIA DOS RESULTADOS
# Vamos verificar o que sobrou após o algoritmo tentar "costurar" as piscadas.
perdas_finais = df['Gaze X'].isna().sum()
linhas_recuperadas = perdas_iniciais - perdas_finais

print("\n--- 🛠️ RELATÓRIO DE LIMPEZA METODOLÓGICA ---")
print(f"Total de falhas detectadas (Pupila = 0): {perdas_iniciais} frames.")
print(f"Frames recuperados por interpolação de piscadas (<= 400ms): {linhas_recuperadas} frames.")
print(f"Perdas definitivas (buracos longos, track loss > 400ms): {perdas_finais} frames.")

# 6. SALVAMENTO
df.to_csv(caminho_saida, index=False)
print(f"\n✅ Arquivo limpo e estruturado salvo em: {caminho_saida}")

Iniciando a Fase 2: Limpeza e Interpolação Fisiológica...

--- 🛠️ RELATÓRIO DE LIMPEZA METODOLÓGICA ---
Total de falhas detectadas (Pupila = 0): 103 frames.
Frames recuperados por interpolação de piscadas (<= 400ms): 103 frames.
Perdas definitivas (buracos longos, track loss > 400ms): 0 frames.

✅ Arquivo limpo e estruturado salvo em: ../data/csv/T43_CONTROLE_limpo.csv


In [1]:
import pandas as pd
import numpy as np

# 1. CARREGAMENTO DOS DADOS LIMPOS DA FASE 2
caminho_entrada = '../data/csv/T43_CONTROLE_limpo.csv'
caminho_saida = '../data/csv/T43_CONTROLE_features.csv'

print("Iniciando a Fase 3: Algoritmo de Classificação I-VT (Velocity Threshold)...")
df = pd.read_csv(caminho_entrada)

# 2. CÁLCULO DE DISTÂNCIA E VELOCIDADE (A Física do Olho)
# Usamos o Teorema de Pitágoras para calcular a distância percorrida em pixels 
# entre a linha atual e a linha anterior.
df['Delta_X'] = df['Gaze X'].diff()
df['Delta_Y'] = df['Gaze Y'].diff()
df['Distancia_px'] = np.sqrt(df['Delta_X']**2 + df['Delta_Y']**2)

# Calculamos a velocidade (Pixels por Segundo)
# Usamos a coluna 'Timegap (ms)' para saber o tempo exato entre as capturas
df['Velocidade_px_s'] = (df['Distancia_px'] / df['Timegap (ms)']) * 1000

# 3. APLICAÇÃO DO LIMIAR DE VELOCIDADE (O Filtro Cognitivo)
# A literatura geralmente define sacadas como movimentos mais rápidos que ~300 graus/segundo.
# Em pixels (variando pela resolução da tela e distância do sujeito), um limiar seguro 
# para iniciar a separação é de 250 a 300 pixels por segundo.
LIMIAR_VELOCIDADE = 250

# Criamos uma nova coluna chamada 'Evento_Cognitivo' e preenchemos tudo como 'Indefinido'
df['Evento_Cognitivo'] = 'Indefinido'

# Regra A: Se a velocidade for MENOR que o limiar -> Fixação
df.loc[df['Velocidade_px_s'] <= LIMIAR_VELOCIDADE, 'Evento_Cognitivo'] = 'Fixação'

# Regra B: Se a velocidade for MAIOR que o limiar -> Sacada
df.loc[df['Velocidade_px_s'] > LIMIAR_VELOCIDADE, 'Evento_Cognitivo'] = 'Sacada'

# Regra C: Se o Gaze X for nulo (lembra das perdas > 400ms?), o evento é 'Perda de Sinal'
df.loc[df['Gaze X'].isna(), 'Evento_Cognitivo'] = 'Perda de Sinal'

# 4. AUDITORIA E RELATÓRIO
total_linhas = len(df)
contagem_eventos = df['Evento_Cognitivo'].value_counts()

print("\n--- 🧠 RELATÓRIO DE ASSINATURAS COGNITIVAS ---")
print(f"Total de registros analisados: {total_linhas}")
print("\nDistribuição dos eventos:")
for evento, quantidade in contagem_eventos.items():
    porcentagem = (quantidade / total_linhas) * 100
    print(f"- {evento}: {quantidade} frames ({porcentagem:.1f}%)")

# 5. SALVAMENTO DA MATRIZ DE FEATURES
df.to_csv(caminho_saida, index=False)
print(f"\n✅ Matriz de features salva em: {caminho_saida}")

Iniciando a Fase 3: Algoritmo de Classificação I-VT (Velocity Threshold)...

--- 🧠 RELATÓRIO DE ASSINATURAS COGNITIVAS ---
Total de registros analisados: 11061

Distribuição dos eventos:
- Sacada: 10027 frames (90.7%)
- Fixação: 1033 frames (9.3%)
- Indefinido: 1 frames (0.0%)

✅ Matriz de features salva em: ../data/csv/T43_CONTROLE_features.csv


In [2]:
import pandas as pd
import numpy as np

# 1. CARREGAMENTO DOS DADOS LIMPOS
caminho_entrada = '../data/csv/T43_CONTROLE_limpo.csv'
caminho_saida_base = '../data/csv/T43_CONTROLE_features' # Não colocamos .csv ainda, pois salvaremos com o limiar escolhido

print("Iniciando a Fase 3 v2.0: Suavização e Teste de Limiares I-VT...\n")
df = pd.read_csv(caminho_entrada)

# 2. FILTRO DE SUAVIZAÇÃO (SMOOTHING) PARA REDUZIR JITTER
# Usamos uma janela de 3 frames (~50ms) centralizada para não atrasar o sinal.
# min_periods=1 garante que as bordas ou vizinhos de NaN não sejam perdidos.
df['Gaze X_Smooth'] = df['Gaze X'].rolling(window=3, center=True, min_periods=1).mean()
df['Gaze Y_Smooth'] = df['Gaze Y'].rolling(window=3, center=True, min_periods=1).mean()

# 3. CÁLCULO DE FÍSICA CORRIGIDO (Usando os dados suavizados)
df['Delta_X'] = df['Gaze X_Smooth'].diff()
df['Delta_Y'] = df['Gaze Y_Smooth'].diff()
df['Distancia_px'] = np.sqrt(df['Delta_X']**2 + df['Delta_Y']**2)

# Evita divisão por zero caso o Timegap seja 0 em algum frame anômalo
df['Timegap (ms)'] = df['Timegap (ms)'].replace(0, np.nan)
df['Velocidade_px_s'] = (df['Distancia_px'] / df['Timegap (ms)']) * 1000

# 4. CALIBRADOR DE LIMIARES (Threshold Sweep)
limiares_para_testar = [250, 500, 750, 1000]
total_linhas = len(df)

print("--- 🧠 TESTE COMPARATIVO DE LIMIARES DE VELOCIDADE ---")
print("Objetivo Fisiológico: Encontrar o limiar que resulte entre 70% e 90% de Fixações.\n")

for limiar in limiares_para_testar:
    # Cria uma coluna temporária para este teste
    col_teste = f'Evento_{limiar}'
    df[col_teste] = 'Indefinido'
    
    # Aplica as regras
    df.loc[df['Velocidade_px_s'] <= limiar, col_teste] = 'Fixação'
    df.loc[df['Velocidade_px_s'] > limiar, col_teste] = 'Sacada'
    df.loc[df['Gaze X'].isna(), col_teste] = 'Perda de Sinal'
    
    # Calcula as porcentagens
    fixacoes = (df[col_teste] == 'Fixação').sum()
    sacadas = (df[col_teste] == 'Sacada').sum()
    pct_fix = (fixacoes / total_linhas) * 100
    pct_sac = (sacadas / total_linhas) * 100
    
    print(f"Limiar de {limiar} px/s  ->  Fixações: {pct_fix:.1f}%  |  Sacadas: {pct_sac:.1f}%")

print("\n-----------------------------------------------------------")
print("AGUARDANDO DECISÃO DO PESQUISADOR:")
print("Analise as porcentagens acima. Qual dos limiares chegou mais perto da métrica esperada (70-90% de fixação)?")

Iniciando a Fase 3 v2.0: Suavização e Teste de Limiares I-VT...

--- 🧠 TESTE COMPARATIVO DE LIMIARES DE VELOCIDADE ---
Objetivo Fisiológico: Encontrar o limiar que resulte entre 70% e 90% de Fixações.

Limiar de 250 px/s  ->  Fixações: 41.9%  |  Sacadas: 58.1%
Limiar de 500 px/s  ->  Fixações: 70.9%  |  Sacadas: 29.1%
Limiar de 750 px/s  ->  Fixações: 79.3%  |  Sacadas: 20.6%
Limiar de 1000 px/s  ->  Fixações: 83.3%  |  Sacadas: 16.7%

-----------------------------------------------------------
AGUARDANDO DECISÃO DO PESQUISADOR:
Analise as porcentagens acima. Qual dos limiares chegou mais perto da métrica esperada (70-90% de fixação)?


In [3]:
import pandas as pd
import numpy as np

# 1. CARREGAMENTO
caminho_entrada = '../data/csv/T43_CONTROLE_limpo.csv'
caminho_saida = '../data/csv/T43_CONTROLE_features.csv'

print("Iniciando a Fase 3: Extração Definitiva de Features (I-VT)...")
df = pd.read_csv(caminho_entrada)

# 2. SUAVIZAÇÃO (Janela de 3 frames)
df['Gaze X_Smooth'] = df['Gaze X'].rolling(window=3, center=True, min_periods=1).mean()
df['Gaze Y_Smooth'] = df['Gaze Y'].rolling(window=3, center=True, min_periods=1).mean()

# 3. FÍSICA E VELOCIDADE
df['Delta_X'] = df['Gaze X_Smooth'].diff()
df['Delta_Y'] = df['Gaze Y_Smooth'].diff()
df['Distancia_px'] = np.sqrt(df['Delta_X']**2 + df['Delta_Y']**2)

df['Timegap (ms)'] = df['Timegap (ms)'].replace(0, np.nan)
df['Velocidade_px_s'] = (df['Distancia_px'] / df['Timegap (ms)']) * 1000

# 4. APLICAÇÃO DO LIMIAR OFICIAL (750 px/s)
LIMIAR_OFICIAL = 750
df['Evento_Cognitivo'] = 'Indefinido'

df.loc[df['Velocidade_px_s'] <= LIMIAR_OFICIAL, 'Evento_Cognitivo'] = 'Fixação'
df.loc[df['Velocidade_px_s'] > LIMIAR_OFICIAL, 'Evento_Cognitivo'] = 'Sacada'
df.loc[df['Gaze X'].isna(), 'Evento_Cognitivo'] = 'Perda de Sinal'

# 5. LIMPEZA DA MATRIZ (Removendo colunas de cálculo intermediário para poupar memória)
colunas_para_remover = ['Gaze X_Smooth', 'Gaze Y_Smooth', 'Delta_X', 'Delta_Y']
df = df.drop(columns=colunas_para_remover)

# 6. EXPORTAÇÃO
df.to_csv(caminho_saida, index=False)
print(f"✅ Matriz final de features salva com sucesso em: {caminho_saida}")
print(f"O arquivo agora contém a coluna 'Evento_Cognitivo' calibrada a {LIMIAR_OFICIAL} px/s.")

Iniciando a Fase 3: Extração Definitiva de Features (I-VT)...
✅ Matriz final de features salva com sucesso em: ../data/csv/T43_CONTROLE_features.csv
O arquivo agora contém a coluna 'Evento_Cognitivo' calibrada a 750 px/s.


In [4]:
import pandas as pd
import numpy as np

# 1. CARREGAMENTO DOS DADOS (O resultado da Fase 3)
caminho_entrada = '../data/csv/T43_CONTROLE_features.csv'
caminho_ml_vector = '../data/csv/ML_Dataset_Global.csv'

print("Iniciando a Fase 4: Vetorização de Features para Machine Learning...\n")
df = pd.read_csv(caminho_entrada)

# 2. AGRUPAMENTO DE EVENTOS DISCRETOS (A Mágica da Compressão)
# O código abaixo cria um "ID" único para cada bloco contínuo do mesmo evento.
# Ex: Se houver 10 frames de Fixação seguidos, todos recebem o mesmo ID de bloco.
df['Bloco_ID'] = (df['Evento_Cognitivo'] != df['Evento_Cognitivo'].shift()).cumsum()

# Somamos o tempo (Timegap) dentro de cada bloco para descobrir a duração real daquele evento
eventos_df = df.groupby(['Bloco_ID', 'Evento_Cognitivo'])['Timegap (ms)'].sum().reset_index()
eventos_df.rename(columns={'Timegap (ms)': 'Duracao_Evento_ms'}, inplace=True)

# 3. EXTRAÇÃO DAS MÉTRICAS GLOBAIS DO SUJEITO (As Variáveis Independentes - X)
# Filtramos apenas as Fixações
fixacoes = eventos_df[eventos_df['Evento_Cognitivo'] == 'Fixação']
# Filtramos apenas as Sacadas do arquivo original para tirar a velocidade
sacadas_frames = df[df['Evento_Cognitivo'] == 'Sacada']

# Calculamos a estatística descritiva
qtd_fixacoes = len(fixacoes)
media_duracao_fixacao = fixacoes['Duracao_Evento_ms'].mean()
desvio_duracao_fixacao = fixacoes['Duracao_Evento_ms'].std()
velocidade_media_sacada = sacadas_frames['Velocidade_px_s'].mean()

# Razão de tempo (Quanto tempo o sujeito passou processando vs. buscando informação)
tempo_total_fixacao = fixacoes['Duracao_Evento_ms'].sum()
tempo_total_experimento = df['Timegap (ms)'].sum()
taxa_fixacao_total = (tempo_total_fixacao / tempo_total_experimento) * 100

# 4. MONTAGEM DO VETOR (A linha única do paciente)
# Extraímos a classe do nome do arquivo (CONTROLE ou TEA)
id_sujeito = 'T43'
classe_target = 0 # 0 para Controle, 1 para TEA (A Variável Dependente - Y)

vetor_sujeito = {
    'Subject_ID': id_sujeito,
    'Total_Fixacoes': qtd_fixacoes,
    'Media_Duracao_Fixacao_ms': round(media_duracao_fixacao, 2),
    'SD_Duracao_Fixacao_ms': round(desvio_duracao_fixacao, 2),
    'Velocidade_Media_Sacada_px_s': round(velocidade_media_sacada, 2),
    'Taxa_Tempo_Fixacao_%': round(taxa_fixacao_total, 2),
    'Target_Class': classe_target
}

# Transformamos o dicionário em um DataFrame de 1 linha
df_ml = pd.DataFrame([vetor_sujeito])

print("--- 🧠 VETOR DE CARACTERÍSTICAS (FEATURE VECTOR) ---")
for chave, valor in vetor_sujeito.items():
    print(f"{chave}: {valor}")

# 5. SALVAMENTO (Append Mode)
# Como este arquivo será a base final, se ele não existir, nós criamos. 
# Se existir, nós adicionamos a linha no final (modo 'a').
import os
if not os.path.exists(caminho_ml_vector):
    df_ml.to_csv(caminho_ml_vector, index=False)
    print(f"\n✅ Base de ML criada! Vetor salvo em: {caminho_ml_vector}")
else:
    df_ml.to_csv(caminho_ml_vector, mode='a', header=False, index=False)
    print(f"\n✅ Vetor adicionado à base de ML existente em: {caminho_ml_vector}")

Iniciando a Fase 4: Vetorização de Features para Machine Learning...

--- 🧠 VETOR DE CARACTERÍSTICAS (FEATURE VECTOR) ---
Subject_ID: T43
Total_Fixacoes: 778
Media_Duracao_Fixacao_ms: 213.42
SD_Duracao_Fixacao_ms: 251.51
Velocidade_Media_Sacada_px_s: 2744.29
Taxa_Tempo_Fixacao_%: 79.94
Target_Class: 0

✅ Base de ML criada! Vetor salvo em: ../data/csv/ML_Dataset_Global.csv
